# Fine-Tuning Qwen2.5-0.5B for Ecosystem Intent, Skill Routing & Task Time Allocation

This notebook fine-tunes **`Qwen2.5-0.5B-Instruct`** using **Unsloth** and LoRA to serve as the ultra-fast local router for `adaptive-workflow` and `Fleet-Orchestrator`.

### Capabilities Trained:
1. **Lifecycle Archetype & Skill Matrix Routing** (`ENGINEERING_DEV`, `RESEARCH_ACADEMIC`, `DOMAIN_HARDWARE`, `FRONTEND_PRODUCT`, `SWARM_ORCHESTRATION`, `SYSADMIN_SECURITY`).
2. **Task Time Allocation**:
   - `tier`: Discrete time complexity tier (`T0_MICRO`, `T1_FAST`, `T2_MEDIUM`, `T3_LONG`, `T4_EPIC`)
   - `estimated_duration_s`: Empirical wall-clock duration estimate
   - `timeout_ceiling_s`: Dynamic worker timeout ceiling for `copilot_queue_worker.py`
   - `cpm_weight`: Normalized duration factor ($D_j$) for mathematical Critical Path Method scheduling in `sim/adaptive_orchestrator.py`
   - `execution_route`: Dynamic routing target (`DIRECT_FAST`, `SUBAGENT`, `FLEET_WORKER`, `HUMAN_CONFIRM`)

### Runtime Setting (Colab Pro):
- **Hardware Accelerator**: Select **L4 GPU** (Ada Lovelace, 24GB VRAM) or **T4 GPU**.
- Total training time on L4: **~60 to 90 seconds** (for 1,553 samples).
- Output: Quantized **`q4_k_m` GGUF** model file (~380 MB).

In [3]:
!pip install --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "trl<0.9.0" peft accelerate bitsandbytes

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-u9qs4gwn/unsloth_7e64133091db4395aaa9d9e162565a2a
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-u9qs4gwn/unsloth_7e64133091db4395aaa9d9e162565a2a
  Resolved https://github.com/unslothai/unsloth.git to commit 6bd164faafbc2aaa307b12e5ee16038ca5e27d1b
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 300.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 311.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 311.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 383.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 258.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 354.7 MB/s eta 0:00:00
 

## 1. Load Model & Tokenizer in 4-bit

In [4]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 1024
dtype = None  # Auto-detects bfloat16 on L4 / A100
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-0.5B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.10.3: Fast Qwen2 patching. Transformers: 5.17.0.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 8.9. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Unsloth 2026.10.3 patched 24 layers with 24 QKV layers, 24 O layers and 24 MLP layers.


## 2. Ingest Dataset
Upload `task_time_train.jsonl` and `task_time_eval.jsonl` (from `Fleet-Orchestrator/dataset/`) via the Colab files sidebar.

In [5]:
import os
from datasets import load_dataset

# Determine filenames (supporting task_time or intent_routing naming)
train_file = "task_time_train.jsonl" if os.path.exists("task_time_train.jsonl") else "intent_routing_train.jsonl"
eval_file = "task_time_eval.jsonl" if os.path.exists("task_time_eval.jsonl") else "intent_routing_eval.jsonl"

dataset = load_dataset("json", data_files={
    "train": train_file,
    "eval": eval_file
})

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input_text, output in zip(instructions, inputs, outputs):
        text = f"<|im_start|>system\n{instruction}<|im_end|>\n<|im_start|>user\n{input_text}<|im_end|>\n<|im_start|>assistant\n{output}<|im_end|>"
        texts.append(text)
    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched = True)
print(f"Loaded {len(dataset['train'])} train samples, {len(dataset['eval'])} eval samples from {train_file}")

Generating train split: 0 examples [00:00, ? examples/s]

Generating eval split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/1553 [00:00<?, ? examples/s]

Map:   0%|          | 0/388 [00:00<?, ? examples/s]

Loaded 1553 train samples, 388 eval samples from task_time_train.jsonl


## 3. Train LoRA Adapters (Optimized for L4 / T4)

In [9]:
from trl import SFTTrainer
from transformers import TrainingArguments, Trainer
import unsloth.models._utils

# Compatibility bridge between trl < 0.9.0 and transformers >= 4.47
if hasattr(unsloth.models._utils, "_original_trainer_init"):
    _orig_unsloth_init = unsloth.models._utils._original_trainer_init
    def _patched_unsloth_init(self, *args, **kwargs):
        if "tokenizer" in kwargs and "processing_class" not in kwargs:
            tok = kwargs.pop("tokenizer")
            kwargs["processing_class"] = tok
            self.tokenizer = tok
        elif "tokenizer" in kwargs:
            self.tokenizer = kwargs.pop("tokenizer")
        return _orig_unsloth_init(self, *args, **kwargs)
    unsloth.models._utils._original_trainer_init = _patched_unsloth_init

_orig_trainer_init = Trainer.__init__
def _patched_trainer_init(self, *args, **kwargs):
    if "tokenizer" in kwargs and "processing_class" not in kwargs:
        tok = kwargs.pop("tokenizer")
        kwargs["processing_class"] = tok
        self.tokenizer = tok
    elif "tokenizer" in kwargs:
        self.tokenizer = kwargs.pop("tokenizer")
    return _orig_trainer_init(self, *args, **kwargs)
Trainer.__init__ = _patched_trainer_init

# L4 GPU has 24GB VRAM: use batch size 8 with bf16 for maximum speed
has_bf16 = torch.cuda.is_bf16_supported()
batch_size = 8 if has_bf16 else 4
grad_accum = 2 if has_bf16 else 4

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset["train"],
    eval_dataset = dataset["eval"],
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = batch_size,
        gradient_accumulation_steps = grad_accum,
        warmup_steps = 10,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        fp16 = not has_bf16,
        bf16 = has_bf16,
        logging_steps = 15,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 42,
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,553 | Num Epochs = 3 | Total steps = 294
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 2 x 1) = 16
 "-____-"     Trainable parameters = 8,798,208 of 502,830,976 (1.75% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
15,3.001636
30,1.048693
45,0.923080
60,0.853323
75,0.930786
90,1.074843
105,0.773208
120,0.759539
135,1.018750
150,0.856024


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-294/tokenizer_config.json.


## 4. Test Structured Inference (Intent + Time Allocation)

In [10]:
import json
FastLanguageModel.for_inference(model)

sys_prompt = "You are the high-speed Intent, Skill, and Task Time Allocation Router for the Aaradhya development ecosystem. Classify the incoming user intent into the exact lifecycle archetype, tier, 2D matrix cell, primary skill, supporting skills, velocity profile, and task time allocation (tier, estimated_duration_s, timeout_ceiling_s, cpm_weight, execution_route) in strict JSON format."

test_prompts = [
    "Fix the regression in test_warehouse_mem_sim.py where queue latency was calculating as zero.",
    "Scaffold IOE BE semester IV-II notes for CE 752 and EX 756 with syllabus markdown hubs.",
    "Dispatch autonomous fleet swarm across 27 workers to implement parallel unit tests.",
    "Add a new project card to access.js and re-encrypt with AES-256-GCM.",
    "Audit system memory and check git status."
]

for prompt in test_prompts:
    prompt_text = f"<|im_start|>system\n{sys_prompt}<|im_end|>\n<|im_start|>user\n{prompt}<|im_end|>\n<|im_start|>assistant\n"
    inputs = tokenizer([prompt_text], return_tensors = "pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens = 220, use_cache = True)
    res = tokenizer.batch_decode(outputs)[0].split("<|im_start|>assistant\n")[-1].replace("<|im_end|>", "").strip()
    print(f"\n{'='*70}")
    print(f"Prompt: {prompt}")
    try:
        parsed = json.loads(res)
        print(json.dumps(parsed, indent=2))
    except Exception:
        print(f"Raw Output:\n{res}")

Both `max_new_tokens` (=220) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=220) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Prompt: Fix the regression in test_warehouse_mem_sim.py where queue latency was calculating as zero.
{
  "archetype": "SWARM_ORCHESTRATION",
  "tier": "Tier 2",
  "matrix_cell": "(V2, R0)",
  "policy": "FLEET_SWARM",
  "primary_skill": "fleet-orchestrator",
  "supporting_skills": [
    "adaptive-workflow"
  ],
  "velocity": "TURBO",
  "time_allocation": {
    "tier": "T1_FAST",
    "estimated_duration_s": 45,
    "timeout_ceiling_s": 100,
    "cpm_weight": 1.5,
    "execution_route": "FLEET_WORKER"
  }
}


Both `max_new_tokens` (=220) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Prompt: Scaffold IOE BE semester IV-II notes for CE 752 and EX 756 with syllabus markdown hubs.
{
  "archetype": "SYSADMIN_SECURITY",
  "tier": "Tier 1",
  "matrix_cell": "(V0, R1)",
  "policy": "BRANCH_GUARD",
  "primary_skill": "win-vault",
  "supporting_skills": [],
  "velocity": "BALANCED",
  "time_allocation": {
    "tier": "T3_LONG",
    "estimated_duration_s": 489,
    "timeout_ceiling_s": 1091,
    "cpm_weight": 16.29,
    "execution_route": "SUBAGENT"
  }
}


Both `max_new_tokens` (=220) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Prompt: Dispatch autonomous fleet swarm across 27 workers to implement parallel unit tests.
{
  "archetype": "SWARM_ORCHESTRATION",
  "tier": "Tier 2",
  "matrix_cell": "(V2, R0)",
  "policy": "FLEET_SWARM",
  "primary_skill": "fleet-orchestrator",
  "supporting_skills": [
    "adaptive-workflow"
  ],
  "velocity": "TURBO",
  "time_allocation": {
    "tier": "T1_FAST",
    "estimated_duration_s": 45,
    "timeout_ceiling_s": 100,
    "cpm_weight": 1.5,
    "execution_route": "DIRECT_FAST"
  }
}


Both `max_new_tokens` (=220) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Prompt: Add a new project card to access.js and re-encrypt with AES-256-GCM.
{
  "archetype": "ENGINEERING_DEV",
  "tier": "Tier 1",
  "matrix_cell": "(V0, R0)",
  "policy": "DIRECT_FAST",
  "primary_skill": "github-workflow",
  "supporting_skills": [],
  "velocity": "BALANCED",
  "time_allocation": {
    "tier": "T3_LONG",
    "estimated_duration_s": 489,
    "timeout_ceiling_s": 1070,
    "cpm_weight": 16.29,
    "execution_route": "SUBAGENT"
  }
}

Prompt: Audit system memory and check git status.
{
  "archetype": "ENGINEERING_DEV",
  "tier": "Tier 1",
  "matrix_cell": "(V0, R0)",
  "policy": "DIRECT_FAST",
  "primary_skill": "github-workflow",
  "supporting_skills": [],
  "velocity": "BALANCED",
  "time_allocation": {
    "tier": "T1_FAST",
    "estimated_duration_s": 37,
    "timeout_ceiling_s": 82,
    "cpm_weight": 1.24,
    "execution_route": "SUBAGENT"
  }
}


## 5. Export Quantized INT4 GGUF (Q4_K_M) & Download
Quantizes the model directly to `q4_k_m` GGUF (~380 MB), ready to mount into LM Studio (`localhost:1234`) or llama.cpp.

In [11]:
import glob
from google.colab import files

# Export quantized GGUF (~380 MB)
model.save_pretrained_gguf("qwen_intent_router_q4", tokenizer, quantization_method = "q4_k_m")

gguf_files = glob.glob("qwen_intent_router_q4/*.gguf")
if gguf_files:
    target_gguf = gguf_files[0]
    print(f"Quantized GGUF ready: {target_gguf}")
    print("Triggering browser download...")
    files.download(target_gguf)
else:
    print("Error: GGUF file not found in qwen_intent_router_q4/")

Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/761 [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in qwen_intent_router_q4/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [00:03<00:00,  3.14s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:03<00:00,  3.86s/it]


Unsloth: Merge process complete. Saved to `/content/qwen_intent_router_q4`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF bf16 might take 3 minutes.
\        /    [2] Converting GGUF bf16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Installing prebuilt llama.cpp b11505-mix-5209c9f (app-b11505-mix-5209c9f-linux-x64-cpu.tar.gz) - skipping compilation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into bf16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['qwen_intent_router_q4_gguf/Qwen2.5-0.5B-Instruct.BF16.gguf']
Unsloth: [2] Converting GGUF bf16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF con

## (Optional) Direct Save to Google Drive
Mount your Google Drive to save the model directly without browser download delays.

In [14]:
import os
import shutil
from google.colab import drive

# 1. Mount Google Drive if not already mounted
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

# 2. Source file generated by Unsloth
source_file = "qwen_intent_router_q4_gguf/Qwen2.5-0.5B-Instruct.Q4_K_M.gguf"

# 3. Destination folder in your Google Drive
target_dir = "/content/drive/MyDrive/Share to Aaradhya/Super-NLM/Fleet-Orchestrator"
os.makedirs(target_dir, exist_ok=True)

dest_path = os.path.join(target_dir, "qwen_intent_router_q4_k_m.gguf")

print(f"Copying {source_file} -> {dest_path}...")
shutil.copy(source_file, dest_path)
print(f" Successfully copied! File size: {os.path.getsize(dest_path) / (1024*1024):.1f} MB")

Copying qwen_intent_router_q4_gguf/Qwen2.5-0.5B-Instruct.Q4_K_M.gguf -> /content/drive/MyDrive/Share to Aaradhya/Super-NLM/Fleet-Orchestrator/qwen_intent_router_q4_k_m.gguf...
 Successfully copied! File size: 379.4 MB
